In [1]:
!pip install "soup-cli[train]"

In [2]:
from datasets import load_dataset

ds = load_dataset("multilingual/orca_dpo_pairs", split="ru_train")

small_ds = ds.select(range(500))

print(f"Записей: {len(small_ds)}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/ar_train-00000-of-00001.parquet:   0%|          | 0.00/6.53M [00:00<?, ?B/s]

data/cn_train-00000-of-00001.parquet:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

data/de_train-00000-of-00001.parquet:   0%|          | 0.00/20.6M [00:00<?, ?B/s]

...

Generating es_train split:   0%|          | 0/11687 [00:00<?, ? examples/s]

Generating fr_train split:   0%|          | 0/11104 [00:00<?, ? examples/s]

Generating ru_train split:   0%|          | 0/11687 [00:00<?, ? examples/s]

Generating tr_train split:   0%|          | 0/3402 [00:00<?, ? examples/s]

Записей: 500


In [3]:
if 'question' in small_ds.column_names:
    small_ds = small_ds.rename_column("question", "prompt")

small_ds.to_json("ru_dpo_500.jsonl")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

7082313

In [4]:
!soup data   inspect ru_dpo_500.jsonl

Inspecting ru_dpo_500.jsonl...

                            Dataset Stats                            
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric             ┃ Value                                        ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Total samples      │ 500                                          │
│ Columns            │ mllm_index, system, prompt, chosen, rejected │
│ Avg length (chars) │ 2826                                         │
│ Min length         │ 289                                          │
│ Max length         │ 19793                                        │
│ Empty fields       │ 41                                           │
│ Duplicates         │ 0                                            │
└────────────────────┴──────────────────────────────────────────────┘

Sample rows (5):
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ mllm_index ┃ system        

In [14]:
import json
rows=[json.loads(l) for l in open("ru_dpo_500.jsonl", encoding="utf-8") if l.strip()]
for k in ["system","prompt","chosen","rejected"]:
    print(k, sum(1 for r in rows if not r.get(k)))
print("chosen==rejected:", sum(r["chosen"]==r["rejected"] for r in rows))
print("rejected 'Конечно':", sum(str(r["rejected"]).startswith("Конечно") for r in rows), "chosen:", sum(str(r["chosen"]).startswith("Конечно") for r in rows))
print("avg len chosen/rejected:", sum(len(str(r["chosen"])) for r in rows)/len(rows), sum(len(str(r["rejected"])) for r in rows)/len(rows))

system 41
prompt 0
chosen 0
rejected 0
chosen==rejected: 0
rejected 'Конечно': 239 chosen: 2
avg len chosen/rejected: 726.202 1099.75


In [5]:
!soup --help

                                                                                
 Usage: soup [OPTIONS] [COMMAND] [ARGS]...                                      
                                                                                
 Fine-tune and post-train LLMs in one command. No SSH, no config hell.          
                                                                                
 GitHub: https://github.com/MakazhanAlpamys/Soup                                
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --verbose             -V            Show full traceback on errors            │
│ --log-level                   TEXT  Logging tier: quiet | normal | verbose | │
│                                     debug                                    │
│                                     [default: normal]                        │
│ --no-audit-log            

In [6]:
import os
from huggingface_hub import login

#Access Token from HF
login("HF_token")

In [7]:
!soup doctor

Soup Doctor - checking your environment...

╭─────────────────────────────────── System ───────────────────────────────────╮
│ Python:   3.13.15                                                            │
│ Platform: Linux 6.6.122+                                                     │
│ Arch:     x86_64                                                             │
│ `soup` runs under /usr/bin/python3; `python` on your PATH is                 │
│ /usr/local/bin/python. site-packages may differ — for any `python -c` check  │
│ use the soup interpreter explicitly.                                         │
╰──────────────────────────────────────────────────────────────────────────────╯
╭──────────────────────────────────── GPU ─────────────────────────────────────╮
│ CUDA:     available (v12.8)                                                  │
│ GPUs:     1                                                                  │
│   GPU 0: Tesla T4 (14.6 GB)                                    

In [8]:
!soup init

╭──────────────────────────────────────────────────────────────────────────────╮
│ Soup Config Wizard                                                           │
╰───────────────────────── Let's set up your training ─────────────────────────╯
Base model (meta-llama/Llama-3.1-8B-Instruct): 
Task [sft/dpo/kto/orpo/simpo/ipo/grpo/ppo/reward_model/pretrain/embedding] 
(sft): dpo
Training data path (./data/train.jsonl): ru_dpo_500.jsonl
Epochs (3): 
Use QLoRA (4-bit)? [yes/no] (yes): 
╭─────────────────────────────────── Ready! ───────────────────────────────────╮
│ Config saved to soup.yaml                                                    │
│                                                                              │
│ Next step: soup train --config soup.yaml                                     │
╰──────────────────────────────────────────────────────────────────────────────╯


In [10]:
!soup train --config soup.yaml

Loading config from soup.yaml...
╭─────────────────────────────── Training Setup ───────────────────────────────╮
│ Device:  Tesla T4                                                            │
│ Memory:  14.6 GB                                                             │
│ Model:   meta-llama/Llama-3.1-8B-Instruct                                    │
│ Task:    dpo                                                                 │
│ Backend: transformers                                                        │
│ LoRA:    r=64, alpha=16                                                      │
│ Quant:   4bit                                                                │
╰──────────────────────────────────────────────────────────────────────────────╯
Start training? [Y/n]: y
Loading dataset...
Loaded: 450 train samples
Run ID: run_20260930_090307_d27a0e88
Setting up model + trainer...
2026-09-30 09:03:10.851054: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow bina

In [11]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [12]:
!pip install bitsandbytes

In [13]:
!mkdir -p logs
!nohup nvidia-smi --query-gpu=timestamp,memory.used,memory.total,utilization.gpu --format=csv -l 5 > logs/nvidia_smi_timeline.csv 2>&1 &

In [15]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import DPOTrainer
from datasets import Dataset

# 1. Сброс памяти
torch.cuda.empty_cache()

# 2. Загрузка подготовленного датасета (построчное чтение JSONL)
data = []
with open("ru_dpo_500.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

ds = Dataset.from_list(data)
print(f"Успешно загружено записей: {len(ds)}")

# 3. Загрузка модели и токенизатора с 4-битным квантованием (QLoRA)
model_id = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Конфигурация для сжатия модели, чтобы она влезла в 16GB T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0}
)

# 4. Настройка LoRA
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# Фиксируем веса до начала обучения
initial_lora_weights = {
    k: v.cpu().clone()
    for k, v in model.named_parameters()
    if "lora" in k.lower()
}

from trl import DPOTrainer, DPOConfig

# 5. Параметры обучения DPO
training_args = DPOConfig(
    output_dir="./dpo_output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=5,
    save_strategy="no",
    fp16=True,
    report_to="none",
    beta=0.1,
    max_length=256,
    max_prompt_length=128
)

# 6. Инициализация и запуск DPO Trainer
dpo_trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=ds,
    processing_class=tokenizer
)

import datetime, torch
from transformers import TrainerCallback

class TsLog(TrainerCallback):
    def __init__(self, path="logs/train_timestamped.log"):
        self.f = open(path, "a", buffering=1)
    def on_step_end(self, args, state, control, **kw):
        peak = torch.cuda.max_memory_allocated() / 1024**3
        self.f.write(f"{datetime.datetime.now().isoformat()} step={state.global_step} peak_alloc_GiB={peak:.2f}\n")
    def on_log(self, args, state, control, logs=None, **kw):
        self.f.write(f"{datetime.datetime.now().isoformat()} LOG {logs}\n")

torch.cuda.reset_peak_memory_stats()
dpo_trainer.add_callback(TsLog())
print("=== ЗАПУСК DPO ОБУЧЕНИЯ ===")
dpo_trainer.train()
print("=== ОБУЧЕНИЕ ЗАВЕРШЕНО ===")

Успешно загружено записей: 500


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Extracting prompt in train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


=== ЗАПУСК DPO ОБУЧЕНИЯ ===


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss
5,0.692200
10,0.682100
15,0.654100
20,0.647900
25,0.614300
30,0.604300
35,0.508100
40,0.487000
45,0.417300
50,0.408600


=== ОБУЧЕНИЕ ЗАВЕРШЕНО ===


In [16]:
import torch

# Замер памяти
peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)
print(f"Пиковый расход VRAM PyTorch: {peak_vram:.2f} GB")

# Сырой вывод
!nvidia-smi > logs_nvidia_smi.txt
print("Файл logs_nvidia_smi.txt сохранен.")

# ПРОВЕРКА ОБНОВЛЕНИЯ ВЕСОВ (Для раздела Prove Training)
# Извлекаем веса после обучения
trained_lora_weights = {
    k: v.cpu().clone()
    for k, v in model.named_parameters()
    if "lora" in k.lower()
}

# Считаем L2-норму разницы между initial_lora_weights и trained_lora_weights
total_l2_norm = 0.0
for name, init_param in initial_lora_weights.items():
    if name in trained_lora_weights:
        diff = (trained_lora_weights[name].float() - init_param.float())
        total_l2_norm += torch.norm(diff, p=2).item() ** 2

total_l2_norm = total_l2_norm ** 0.5

print(f"\nL2-норма изменения весов LoRA: {total_l2_norm:.6f}")
if total_l2_norm > 1e-5:
    print("[ПРОЙДЕНО] Градиенты применились, веса адаптера изменились.")
else:
    print("[ОШИБКА] Веса заморожены! Модель не обучалась.")

Пиковый расход VRAM PyTorch: 6.51 GB
Файл logs_nvidia_smi.txt сохранен.

L2-норма изменения весов LoRA: 2.032301
[ПРОЙДЕНО] Градиенты применились, веса адаптера изменились.


In [18]:
!pip freeze > requirements.txt

In [19]:
!zip -r logs.zip logs requirements.txt
from google.colab import files
files.download("logs.zip")

  adding: logs/ (stored 0%)
  adding: logs/train_timestamped.log (deflated 75%)
  adding: logs/nvidia_smi_timeline.csv (deflated 87%)
  adding: requirements.txt (deflated 57%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>